# Model Training Results: Pairwise Amino Acid Classification

This notebook analyzes model training results from 190 pairwise amino acid comparisons.
We compare different model architectures and evaluate the impact of dwell time features.

In [ ]:
import warnings

import plotnine as p9
import polars as pl
from tabulate import tabulate

warnings.filterwarnings("ignore")

# Set plot aesthetics
p9.theme_set(p9.theme_minimal() + p9.theme(figure_size=(12, 6)))

## Load Data

In [ ]:
# Load the pairwise architecture comparison data
data_path = "/scratch/alpine/jhesselberth@xsede.org/leech/synthetic-trna/results/metrics/comparison/aggregate/pairwise_architecture_comparison.tsv"

df = pl.read_csv(data_path, separator="\t")

print(f"Total rows: {len(df)}")
print(f"Total pairs: {df['pair'].n_unique()}")
print(f"Architectures: {df['architecture'].unique().to_list()}")
print(f"\nColumns: {df.columns}")

In [ ]:
# Show first few rows
df.head(10)

## Summary Statistics

In [ ]:
# Overall performance by architecture
summary = (
    df.group_by("architecture")
    .agg(
        [
            pl.col("accuracy_mean").mean().alias("avg_accuracy"),
            pl.col("accuracy_mean").std().alias("std_accuracy"),
            pl.col("f1_mean").mean().alias("avg_f1"),
            pl.col("f1_mean").std().alias("std_f1"),
            pl.col("auroc_mean").mean().alias("avg_auroc"),
            pl.col("auroc_mean").std().alias("std_auroc"),
            pl.count().alias("n_pairs"),
        ]
    )
    .sort("avg_f1", descending=True)
)

print("\nOverall Performance by Architecture (averaged across all pairs):")
print(
    tabulate(
        summary.to_pandas().values,
        headers=summary.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

In [ ]:
# Identify best and worst performing pairs
best_pairs = (
    df.sort("f1_mean", descending=True)
    .select(["pair", "architecture", "f1_mean", "accuracy_mean"])
    .head(10)
)
worst_pairs = (
    df.sort("f1_mean").select(["pair", "architecture", "f1_mean", "accuracy_mean"]).head(10)
)

print("\nTop 10 Best Performing Pairs:")
print(
    tabulate(
        best_pairs.to_pandas().values,
        headers=best_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)
print("\nTop 10 Most Challenging Pairs:")
print(
    tabulate(
        worst_pairs.to_pandas().values,
        headers=worst_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

## Visualization 1: Architecture Performance Comparison

Compare F1 scores across all architectures and amino acid pairs using box plots and violin plots.

In [ ]:
# Convert to pandas for plotnine
df_pd = df.to_pandas()

# Box plot of F1 scores by architecture
p1 = (
    p9.ggplot(df_pd, p9.aes(x="architecture", y="f1_mean", fill="architecture"))
    + p9.geom_boxplot(alpha=0.7)
    + p9.geom_jitter(width=0.2, alpha=0.3, size=1)
    + p9.labs(
        title="Model Architecture Comparison: F1 Score Distribution",
        subtitle=f"Performance across {df['pair'].n_unique()} pairwise amino acid comparisons",
        x="Architecture",
        y="F1 Score (mean)",
    )
    + p9.theme(
        axis_text_x=p9.element_text(rotation=45, hjust=1), legend_position="none", figure_size=(10, 6)
    )
    + p9.scale_fill_brewer(type="qual", palette="Set2")
)

print(p1)

In [ ]:
# Violin plot showing distribution density
p2 = (
    p9.ggplot(df_pd, p9.aes(x="architecture", y="accuracy_mean", fill="architecture"))
    + p9.geom_violin(alpha=0.7)
    + p9.geom_boxplot(width=0.1, alpha=0.5)
    + p9.labs(
        title="Model Architecture Comparison: Accuracy Distribution",
        subtitle="Violin plots show density of performance across pairs",
        x="Architecture",
        y="Accuracy (mean)",
    )
    + p9.theme(
        axis_text_x=p9.element_text(rotation=45, hjust=1), legend_position="none", figure_size=(10, 6)
    )
    + p9.scale_fill_brewer(type="qual", palette="Set1")
)

print(p2)

In [ ]:
# Multi-metric comparison
metrics_long = df_pd.melt(
    id_vars=["pair", "architecture"],
    value_vars=["accuracy_mean", "f1_mean", "auroc_mean", "auprc_mean"],
    var_name="metric",
    value_name="score",
)

# Clean up metric names
metrics_long["metric"] = metrics_long["metric"].str.replace("_mean", "").str.upper()

p3 = (
    p9.ggplot(metrics_long, p9.aes(x="architecture", y="score", fill="architecture"))
    + p9.geom_boxplot(alpha=0.7)
    + p9.facet_wrap("~metric", scales="free_y")
    + p9.labs(
        title="Multi-Metric Performance Comparison Across Architectures",
        x="Architecture",
        y="Score",
    )
    + p9.theme(
        axis_text_x=p9.element_text(rotation=45, hjust=1), legend_position="none", figure_size=(14, 8)
    )
    + p9.scale_fill_brewer(type="qual", palette="Set2")
)

print(p3)

## Visualization 2: Dwell Time Feature Impact

Compare models with and without dwell time features to quantify their impact.
This assumes architectures like `ConvLSTMDwell` (with dwell) vs `ConvLSTMBase` (without dwell).

In [ ]:
# Identify dwell vs non-dwell architectures
architectures = df_pd["architecture"].unique()
print(f"Available architectures: {architectures}")

# Try to identify dwell-enabled vs baseline models
dwell_models = [arch for arch in architectures if "dwell" in arch.lower()]
base_models = [arch for arch in architectures if "base" in arch.lower()]

print(f"\nDwell-enabled models: {dwell_models}")
print(f"Baseline models: {base_models}")

In [ ]:
# If we have both dwell and base models, create direct comparison
if dwell_models and base_models:
    # Create a subset for dwell comparison
    dwell_comparison = df_pd[df_pd["architecture"].isin(dwell_models + base_models)].copy()

    # Categorize as dwell or baseline
    dwell_comparison["feature_type"] = dwell_comparison["architecture"].apply(
        lambda x: "With Dwell Features" if "dwell" in x.lower() else "Baseline (No Dwell)"
    )

    # Calculate improvement for each pair
    if len(dwell_models) == 1 and len(base_models) == 1:
        dwell_arch = dwell_models[0]
        base_arch = base_models[0]

        dwell_df = df_pd[df_pd["architecture"] == dwell_arch][
            ["pair", "f1_mean", "accuracy_mean"]
        ].rename(columns={"f1_mean": "f1_dwell", "accuracy_mean": "acc_dwell"})
        base_df = df_pd[df_pd["architecture"] == base_arch][
            ["pair", "f1_mean", "accuracy_mean"]
        ].rename(columns={"f1_mean": "f1_base", "accuracy_mean": "acc_base"})

        improvement_df = dwell_df.merge(base_df, on="pair")
        improvement_df["f1_improvement"] = improvement_df["f1_dwell"] - improvement_df["f1_base"]
        improvement_df["acc_improvement"] = improvement_df["acc_dwell"] - improvement_df["acc_base"]

        print(f"\nDwell Time Feature Impact ({dwell_arch} vs {base_arch}):")
        print(
            f"Mean F1 improvement: {improvement_df['f1_improvement'].mean():.4f} ± {improvement_df['f1_improvement'].std():.4f}"
        )
        print(
            f"Mean Accuracy improvement: {improvement_df['acc_improvement'].mean():.4f} ± {improvement_df['acc_improvement'].std():.4f}"
        )
        print(
            f"Pairs where dwell helps (F1): {(improvement_df['f1_improvement'] > 0).sum()} / {len(improvement_df)}"
        )
        print(
            f"Pairs where dwell helps (Acc): {(improvement_df['acc_improvement'] > 0).sum()} / {len(improvement_df)}"
        )
else:
    print("Note: Could not identify both dwell and baseline models for direct comparison.")
    dwell_comparison = df_pd.copy()
    dwell_comparison["feature_type"] = dwell_comparison["architecture"]

In [ ]:
# Side-by-side comparison of dwell vs baseline
if dwell_models and base_models:
    p4 = (
        p9.ggplot(dwell_comparison, p9.aes(x="feature_type", y="f1_mean", fill="feature_type"))
        + p9.geom_boxplot(alpha=0.7)
        + p9.geom_jitter(width=0.2, alpha=0.2, size=1)
        + p9.labs(
            title="Impact of Dwell Time Features on Model Performance",
            subtitle="F1 Score distribution with and without dwell time features",
            x="Model Type",
            y="F1 Score (mean)",
        )
        + p9.theme(legend_position="none", figure_size=(10, 6))
        + p9.scale_fill_manual(values=["#E74C3C", "#3498DB"])
    )

    print(p4)

In [ ]:
# Histogram of improvement from dwell features
if dwell_models and base_models and len(dwell_models) == 1 and len(base_models) == 1:
    p5 = (
        p9.ggplot(improvement_df, p9.aes(x="f1_improvement"))
        + p9.geom_histogram(bins=30, fill="#2ECC71", alpha=0.7, color="black")
        + p9.geom_vline(xintercept=0, linetype="dashed", color="red", size=1)
        + p9.labs(
            title="Distribution of F1 Score Improvement from Dwell Time Features",
            subtitle=f"Comparing {dwell_arch} vs {base_arch} across {len(improvement_df)} pairs",
            x="F1 Score Improvement (Dwell - Baseline)",
            y="Count",
        )
        + p9.theme(figure_size=(10, 6))
    )

    print(p5)

    # Show pairs with largest improvements and degradations
    print("\nTop 10 pairs with LARGEST improvement from dwell features:")
    print(
        tabulate(
            improvement_df.nlargest(10, "f1_improvement")[
                ["pair", "f1_base", "f1_dwell", "f1_improvement"]
            ].values,
            headers=["Pair", "F1 Base", "F1 Dwell", "F1 Improvement"],
            tablefmt="grid",
            floatfmt=".4f",
        )
    )

    print("\nTop 10 pairs with LARGEST degradation from dwell features:")
    print(
        tabulate(
            improvement_df.nsmallest(10, "f1_improvement")[
                ["pair", "f1_base", "f1_dwell", "f1_improvement"]
            ].values,
            headers=["Pair", "F1 Base", "F1 Dwell", "F1 Improvement"],
            tablefmt="grid",
            floatfmt=".4f",
        )
    )

In [ ]:
# Scatter plot: baseline vs dwell performance
if dwell_models and base_models and len(dwell_models) == 1 and len(base_models) == 1:
    p6 = (
        p9.ggplot(improvement_df, p9.aes(x="f1_base", y="f1_dwell"))
        + p9.geom_point(alpha=0.6, size=2, color="#9B59B6")
        + p9.geom_abline(intercept=0, slope=1, linetype="dashed", color="red", size=0.8)
        + p9.labs(
            title="Baseline vs Dwell-Enhanced Model Performance",
            subtitle="Points above red line indicate dwell features improve performance",
            x=f"F1 Score - {base_arch}",
            y=f"F1 Score - {dwell_arch}",
        )
        + p9.theme(figure_size=(10, 10))
        + p9.coord_fixed()
    )

    print(p6)

## Top and Bottom Performing Amino Acid Pairs

Identify which amino acid pairs are easiest and hardest to classify.

In [ ]:
# Average performance per pair across all architectures
pair_performance = (
    df.group_by("pair")
    .agg(
        [
            pl.col("f1_mean").mean().alias("avg_f1"),
            pl.col("accuracy_mean").mean().alias("avg_accuracy"),
            pl.col("auroc_mean").mean().alias("avg_auroc"),
        ]
    )
    .sort("avg_f1", descending=True)
)

# Get top and bottom 15
top_pairs = pair_performance.head(15)
bottom_pairs = pair_performance.tail(15)

print("Top 15 Easiest Pairs to Classify:")
print(
    tabulate(
        top_pairs.to_pandas().values,
        headers=top_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)
print("\nTop 15 Hardest Pairs to Classify:")
print(
    tabulate(
        bottom_pairs.to_pandas().values,
        headers=bottom_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

In [ ]:
# Bar plot of top and bottom pairs
extreme_pairs = pl.concat([top_pairs.head(10), bottom_pairs.tail(10)])
extreme_pairs_pd = extreme_pairs.to_pandas()
extreme_pairs_pd["category"] = ["Top 10"] * 10 + ["Bottom 10"] * 10

p7 = (
    p9.ggplot(extreme_pairs_pd, p9.aes(x="reorder(pair, avg_f1)", y="avg_f1", fill="category"))
    + p9.geom_col(alpha=0.8)
    + p9.coord_flip()
    + p9.labs(
        title="Top and Bottom Performing Amino Acid Pairs",
        subtitle="Average F1 score across all model architectures",
        x="Amino Acid Pair",
        y="Average F1 Score",
        fill="Performance",
    )
    + p9.theme(figure_size=(10, 8))
    + p9.scale_fill_manual(values=["#E67E22", "#27AE60"])
)

print(p7)

## Performance Variability Analysis

Examine which pairs have consistent vs variable performance across architectures.

In [ ]:
# Calculate coefficient of variation (std/mean) for each pair across architectures
pair_variability = (
    df.group_by("pair")
    .agg(
        [
            pl.col("f1_mean").mean().alias("avg_f1"),
            pl.col("f1_mean").std().alias("std_f1"),
        ]
    )
    .with_columns((pl.col("std_f1") / pl.col("avg_f1")).alias("cv_f1"))
    .sort("cv_f1", descending=True)
)

print("Top 15 pairs with HIGHEST variability across architectures:")
print(
    tabulate(
        pair_variability.head(15).to_pandas().values,
        headers=pair_variability.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

print("\nTop 15 pairs with LOWEST variability (most consistent):")
print(
    tabulate(
        pair_variability.tail(15).to_pandas().values,
        headers=pair_variability.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

In [ ]:
# Scatter plot: mean performance vs variability
pair_var_pd = pair_variability.to_pandas()

p8 = (
    p9.ggplot(pair_var_pd, p9.aes(x="avg_f1", y="std_f1"))
    + p9.geom_point(alpha=0.6, size=2, color="#E74C3C")
    + p9.labs(
        title="Performance vs Variability Across Amino Acid Pairs",
        subtitle="Each point represents one amino acid pair",
        x="Average F1 Score",
        y="Standard Deviation (across architectures)",
    )
    + p9.theme(figure_size=(10, 6))
)

print(p8)

## Export Summary Statistics

In [ ]:
# Create a comprehensive summary
print("=" * 80)
print("COMPREHENSIVE SUMMARY")
print("=" * 80)
print(f"\nTotal amino acid pairs analyzed: {df['pair'].n_unique()}")
print(f"Total architectures compared: {df['architecture'].n_unique()}")
print(f"Architectures: {', '.join(df['architecture'].unique().to_list())}")

print("\n" + "=" * 80)
print("OVERALL ARCHITECTURE RANKINGS")
print("=" * 80)
print(
    tabulate(
        summary.to_pandas().values,
        headers=summary.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

# Statistical significance of differences (if applicable)
if len(architectures) >= 2:
    print("\n" + "=" * 80)
    print("PERFORMANCE DIFFERENCES")
    print("=" * 80)

    for i, arch1 in enumerate(architectures):
        for arch2 in architectures[i + 1 :]:
            data1 = df_pd[df_pd["architecture"] == arch1]["f1_mean"]
            data2 = df_pd[df_pd["architecture"] == arch2]["f1_mean"]
            diff = data1.mean() - data2.mean()
            print(f"{arch1} vs {arch2}: Δ F1 = {diff:+.4f}")

## Tier 1 Advanced Analyses

The following sections provide in-depth statistical analyses to address:
1. **Hierarchical structure** in architecture and pair performance patterns
2. **Consistency metrics** quantifying robustness and difficulty
3. **Variance decomposition** using two-way ANOVA to identify interaction effects

These analyses help answer:
- Which architectures are most robust across different pair types?
- Which AA pairs show consensus vs variable performance across architectures?
- How much variance is due to architecture choice vs pair difficulty vs interactions?

Reference: Issue #72

In [ ]:
# Additional imports for Tier 1 analyses
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats

# Set seaborn style
sns.set_palette("husl")

### 1. Clustered Heatmap with Hierarchical Clustering

This heatmap shows F1 scores for all AA pairs (rows) vs architectures (columns), with hierarchical clustering applied to both dimensions. Clustering reveals:
- Which AA pairs have similar performance patterns across architectures
- Which architectures perform similarly across different pairs
- Natural groupings and outliers in the performance landscape

In [ ]:
# Pivot data to create matrix: rows=pairs, cols=architectures, values=F1 scores
heatmap_data = df_pd.pivot(index="pair", columns="architecture", values="f1_mean")

# Create clustered heatmap
plt.figure(figsize=(12, 20))
clustergrid = sns.clustermap(
    heatmap_data,
    cmap="RdYlGn",
    vmin=0.5,
    vmax=1.0,
    center=0.75,
    linewidths=0.5,
    figsize=(12, 20),
    dendrogram_ratio=(0.1, 0.2),
    cbar_pos=(0.02, 0.83, 0.03, 0.15),
    xticklabels=True,
    yticklabels=True,
)

# Adjust labels
clustergrid.ax_heatmap.set_xlabel("Architecture", fontsize=12)
clustergrid.ax_heatmap.set_ylabel("Amino Acid Pair", fontsize=12)
clustergrid.ax_heatmap.set_title(
    "F1 Score Heatmap with Hierarchical Clustering\n(Rows and Columns Clustered)",
    fontsize=14,
    pad=20
)

# Rotate x-axis labels
plt.setp(clustergrid.ax_heatmap.get_xticklabels(), rotation=45, ha="right")
plt.setp(clustergrid.ax_heatmap.get_yticklabels(), fontsize=6)

plt.tight_layout()
plt.show()

print(f"\nHeatmap dimensions: {heatmap_data.shape[0]} pairs × {heatmap_data.shape[1]} architectures")

### 2. Consistency Metrics

Quantify consistency and robustness in two ways:

**A. Per AA Pair Metrics** - Which pairs are hardest and which show most variable performance?
- **Mean F1**: Average performance across all architectures
- **Std F1**: Standard deviation (absolute variability)
- **CV (Coefficient of Variation)**: Std/Mean (relative variability)
- **Difficulty Score**: 1 - Mean F1 (higher = harder pair)
- **Consensus Score**: 1 - CV (higher = all architectures agree)

**B. Per Architecture Metrics** - Which architectures are most robust?
- **Mean F1**: Average performance across all pairs
- **Std F1**: Standard deviation across pairs
- **Robustness Score**: 1 / (1 + Std) (higher = more consistent across different pair types)

In [ ]:
# A. Per AA Pair Consistency Metrics
pair_consistency = (
    df.group_by("pair")
    .agg(
        [
            pl.col("f1_mean").mean().alias("mean_f1"),
            pl.col("f1_mean").std().alias("std_f1"),
        ]
    )
    .with_columns(
        [
            (pl.col("std_f1") / pl.col("mean_f1")).alias("cv"),
            (1 - pl.col("f1_mean").mean()).alias("difficulty_score"),
            (1 - (pl.col("std_f1") / pl.col("mean_f1"))).alias("consensus_score"),
        ]
    )
)

# Sort by different criteria to show extremes
print("=" * 80)
print("A. PER AA PAIR CONSISTENCY METRICS")
print("=" * 80)

print("\n--- Top 10 HARDEST Pairs (highest difficulty score) ---")
hardest_pairs = pair_consistency.sort("difficulty_score", descending=True).head(10)
print(
    tabulate(
        hardest_pairs.to_pandas().values,
        headers=hardest_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

print("\n--- Top 10 MOST VARIABLE Pairs (highest CV - architectures disagree most) ---")
most_variable_pairs = pair_consistency.sort("cv", descending=True).head(10)
print(
    tabulate(
        most_variable_pairs.to_pandas().values,
        headers=most_variable_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

print("\n--- Top 10 MOST CONSISTENT Pairs (lowest CV - all architectures agree) ---")
most_consistent_pairs = pair_consistency.sort("cv").head(10)
print(
    tabulate(
        most_consistent_pairs.to_pandas().values,
        headers=most_consistent_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

print("\n--- Top 10 EASIEST Pairs (lowest difficulty score) ---")
easiest_pairs = pair_consistency.sort("difficulty_score").head(10)
print(
    tabulate(
        easiest_pairs.to_pandas().values,
        headers=easiest_pairs.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

In [ ]:
# B. Per Architecture Consistency Metrics
arch_consistency = (
    df.group_by("architecture")
    .agg(
        [
            pl.col("f1_mean").mean().alias("mean_f1"),
            pl.col("f1_mean").std().alias("std_f1"),
            pl.col("f1_mean").min().alias("min_f1"),
            pl.col("f1_mean").max().alias("max_f1"),
            pl.count().alias("n_pairs"),
        ]
    )
    .with_columns(
        [
            (1 / (1 + pl.col("std_f1"))).alias("robustness_score"),
            (pl.col("max_f1") - pl.col("min_f1")).alias("range_f1"),
        ]
    )
    .sort("mean_f1", descending=True)
)

print("\n" + "=" * 80)
print("B. PER ARCHITECTURE CONSISTENCY METRICS")
print("=" * 80)
print("\nRanked by mean F1 (with robustness metrics):")
print("Higher robustness_score = more consistent across different pair types\n")
print(
    tabulate(
        arch_consistency.to_pandas().values,
        headers=arch_consistency.columns,
        tablefmt="grid",
        floatfmt=".4f",
    )
)

# Highlight most robust architecture
most_robust = arch_consistency.sort("robustness_score", descending=True).head(1)
print("\n🏆 Most ROBUST architecture (lowest variance across pairs):")
print(f"   {most_robust['architecture'][0]} (robustness score: {most_robust['robustness_score'][0]:.4f})")

# Highlight best performing architecture
best_overall = arch_consistency.sort("mean_f1", descending=True).head(1)
print("\n🏆 BEST OVERALL architecture (highest mean F1):")
print(f"   {best_overall['architecture'][0]} (mean F1: {best_overall['mean_f1'][0]:.4f})")

### 3. Two-Way ANOVA: Variance Decomposition

Two-way ANOVA decomposes the total variance in F1 scores into components:

1. **Architecture (main effect)**: How much variance is explained by which architecture is used?
2. **AA Pair (main effect)**: How much variance is explained by which pair is being classified?
3. **Architecture × Pair Interaction**: Are there specialization effects where certain architectures excel at specific pairs?

**Interpretation:**
- Large Architecture effect → Choice of architecture matters a lot
- Large Pair effect → Some pairs are universally harder/easier
- Large Interaction effect → Specialization exists (specific architectures excel at specific pair types)
- Compare effect sizes (eta-squared, ω²) to understand relative importance

In [ ]:
# Perform two-way ANOVA
# Check if statsmodels is available, if not use scipy for simpler one-way tests
try:
    from statsmodels.formula.api import ols
    from statsmodels.stats.anova import anova_lm

    # Prepare data for ANOVA
    anova_data = df_pd[["architecture", "pair", "f1_mean"]].copy()

    # Fit the model: F1 ~ Architecture + Pair + Architecture:Pair
    model = ols("f1_mean ~ C(architecture) + C(pair) + C(architecture):C(pair)", data=anova_data).fit()
    anova_table = anova_lm(model, typ=2)

    # Calculate effect sizes (eta-squared)
    anova_table["eta_sq"] = anova_table["sum_sq"] / anova_table["sum_sq"].sum()

    # Calculate omega-squared (more conservative effect size)
    ms_error = anova_table.loc["Residual", "mean_sq"]
    n = len(anova_data)

    def calc_omega_sq(row):
        if row.name == "Residual":
            return np.nan
        df_effect = row["df"]
        ss_effect = row["sum_sq"]
        omega_sq = (ss_effect - df_effect * ms_error) / (anova_table["sum_sq"].sum() + ms_error)
        return max(0, omega_sq)  # omega-squared can't be negative

    anova_table["omega_sq"] = anova_table.apply(calc_omega_sq, axis=1)

    print("=" * 80)
    print("TWO-WAY ANOVA: F1 ~ Architecture + Pair + Architecture:Pair")
    print("=" * 80)
    print("\nANOVA Table:")
    print(
        tabulate(
            anova_table,
            headers=["Source", "Sum Sq", "DF", "F", "P-value", "η² (eta-sq)", "ω² (omega-sq)"],
            tablefmt="grid",
            floatfmt=(".0f", ".4f", ".0f", ".2f", ".4e", ".4f", ".4f"),
        )
    )

    # Interpretation
    print("\n" + "=" * 80)
    print("INTERPRETATION")
    print("=" * 80)

    arch_eta = anova_table.loc["C(architecture)", "eta_sq"]
    pair_eta = anova_table.loc["C(pair)", "eta_sq"]
    interaction_eta = anova_table.loc["C(architecture):C(pair)", "eta_sq"]

    print("\nVariance Explained (η²):")
    print(f"  Architecture:  {arch_eta*100:5.2f}% - {'SMALL' if arch_eta < 0.06 else 'MEDIUM' if arch_eta < 0.14 else 'LARGE'} effect")
    print(f"  AA Pair:       {pair_eta*100:5.2f}% - {'SMALL' if pair_eta < 0.06 else 'MEDIUM' if pair_eta < 0.14 else 'LARGE'} effect")
    print(f"  Interaction:   {interaction_eta*100:5.2f}% - {'SMALL' if interaction_eta < 0.06 else 'MEDIUM' if interaction_eta < 0.14 else 'LARGE'} effect")

    print("\n📊 Key Findings:")
    effects = {
        "Architecture choice": arch_eta,
        "AA pair difficulty": pair_eta,
        "Architecture×Pair specialization": interaction_eta,
    }
    sorted_effects = sorted(effects.items(), key=lambda x: x[1], reverse=True)

    print(f"  1. {sorted_effects[0][0]} explains the MOST variance ({sorted_effects[0][1]*100:.1f}%)")
    print(f"  2. {sorted_effects[1][0]} is second ({sorted_effects[1][1]*100:.1f}%)")
    print(f"  3. {sorted_effects[2][0]} is third ({sorted_effects[2][1]*100:.1f}%)")

    if interaction_eta > 0.06:
        print("\n⚠️  Significant INTERACTION detected!")
        print("   → Some architectures specialize in certain AA pair types")
        print("   → Consider architecture-specific pair recommendations")
    else:
        print("\n✓ Weak interaction effect")
        print("  → Architectures perform consistently across pair types")
        print("  → General architecture rankings are reliable")

    statsmodels_available = True

except ImportError:
    print("statsmodels not available - performing simplified one-way ANOVAs instead")
    print("Install statsmodels for full two-way ANOVA with interactions: pip install statsmodels")

    # Simplified analysis with scipy
    print("\n" + "=" * 80)
    print("SIMPLIFIED VARIANCE ANALYSIS (scipy)")
    print("=" * 80)

    # One-way ANOVA for architecture
    arch_groups = [group["f1_mean"].values for name, group in df_pd.groupby("architecture")]
    f_arch, p_arch = stats.f_oneway(*arch_groups)

    # One-way ANOVA for pair
    pair_groups = [group["f1_mean"].values for name, group in df_pd.groupby("pair")]
    f_pair, p_pair = stats.f_oneway(*pair_groups)

    print(f"\nArchitecture effect: F={f_arch:.2f}, p={p_arch:.4e}")
    print(f"AA Pair effect:      F={f_pair:.2f}, p={p_pair:.4e}")

    print("\nNote: Install statsmodels for full two-way ANOVA with interaction terms")
    statsmodels_available = False

### Summary: Tier 1 Analysis Insights

The three analyses above provide complementary views of the model comparison results:

1. **Clustered Heatmap** → Visual identification of performance patterns and natural groupings
2. **Consistency Metrics** → Quantitative measures of pair difficulty and architecture robustness
3. **Two-Way ANOVA** → Statistical decomposition showing whether variance is driven by architecture, pairs, or interactions

**Next Steps (Future Analyses):**
- **Tier 2**: Architecture specialization analysis, pairwise correlations
- **Tier 3**: Chemical property enrichment, distance-performance relationships
- **Tier 4**: PCA/dimensionality reduction, predictive modeling

See Issue #72 for full analysis roadmap.